# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Setup Colab tu GitHub =====
import os
import sys
import subprocess

from google.colab import drive

drive.mount("/content/drive")

MSSV = "2A202602364"
BRANCH = "submission-2A202602364"
REPO_URL = "https://github.com/n4hhh/K4-Track4-Day1-Neuralnetwork.git"
REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"

if not os.path.isdir(f"{REPO_ROOT}/.git"):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, REPO_ROOT],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", REPO_ROOT, "pull", "--ff-only"],
        check=True,
    )

CODE_DIR = f"{REPO_ROOT}/submission_{MSSV}/code"
OUT_DIR = (
    f"/content/drive/MyDrive/AICB2026/"
    f"K4-Track4-Day1-Neuralnetwork/submission_{MSSV}"
)

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Code directory:", CODE_DIR)
print("PyTorch:", torch.__version__)
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

assert device == "cuda", "Hay chon Runtime > Change runtime type > T4 GPU"

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
from data import prepare_data

subprocess.run(
    [sys.executable, "scripts/split_data.py"],
    cwd=REPO_ROOT,
    check=True,
)

data = prepare_data(
    device=device,
    val_fraction=0.2,
    seed=42,
    processed_dir=f"{REPO_ROOT}/data/processed",
)

assert tuple(data["X_tr"].shape) == (371_847, 54)
assert tuple(data["X_val"].shape) == (92_962, 54)
assert tuple(data["X_eval"].shape) == (116_203, 54)

assert data["X_tr"].dtype == torch.float32
assert data["X_val"].dtype == torch.float32
assert data["X_eval"].dtype == torch.float32
assert data["y_tr"].dtype == torch.int64

for name in ("y_tr", "y_val", "y_eval"):
    counts = torch.bincount(data[name], minlength=7).double()
    proportions = counts / counts.sum()
    print(
        f"{name} class proportions:",
        proportions.cpu().numpy().round(5),
    )

numeric_train = data["X_tr"][:, :10].double()
numeric_mean = numeric_train.mean(dim=0).cpu()
numeric_std = numeric_train.std(dim=0, correction=0).cpu()

binary_values = torch.unique(data["X_tr"][:, 10:]).cpu().tolist()

print("Mean 10 numeric columns:", numeric_mean.numpy().round(6))
print("Std 10 numeric columns:", numeric_std.numpy().round(6))
print("One-hot values:", binary_values)

assert torch.allclose(
    numeric_mean,
    torch.zeros_like(numeric_mean),
    atol=1e-5,
)
assert torch.allclose(
    numeric_std,
    torch.ones_like(numeric_std),
    atol=1e-5,
)
assert binary_values == [0.0, 1.0]

print("Part 0: PASS")

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import torch.nn.functional as F

from model import (
    MLP,
    EXPECTED_PARAMS,
    activation_stats,
    count_params,
)

# Dat seed de ket qua co the lap lai
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


# 1. Kiem tra kien truc va so tham so
model = MLP(
    hidden=(256, 128),
    dropout=0.0,
    init="he",
).to(device)

n_params = count_params(model)

assert n_params == EXPECTED_PARAMS[(256, 128)]
assert n_params == 47_879

print("M-base parameters:", n_params)


# 2. Kiem tra shape logits
model.eval()

with torch.no_grad():
    probe_logits = model(data["X_val"][:8])

assert tuple(probe_logits.shape) == (8, 7)

print("Input shape:", tuple(data["X_val"][:8].shape))
print("Logits shape:", tuple(probe_logits.shape))


# 3. Thong ke kich hoat
stats = activation_stats(
    model,
    data["X_val"][:512],
)

print(
    "Activation std after Linear layers:",
    [round(value, 4) for value in stats],
)


# 4. Loss buoc 0 tren validation
with torch.no_grad():
    step0_logits = model(data["X_val"])
    step0_loss = F.cross_entropy(
        step0_logits,
        data["y_val"],
    ).item()

expected_random_loss = math.log(7)

print(f"Step-0 CE loss: {step0_loss:.4f}")
print(f"ln(7): {expected_random_loss:.4f}")
print(
    "Absolute difference:",
    round(abs(step0_loss - expected_random_loss), 4),
)

assert math.isfinite(step0_loss)


# 5. Kiem tra gradient chay toi moi tham so
model.zero_grad(set_to_none=True)

gradient_logits = model(data["X_tr"][:512])
gradient_loss = F.cross_entropy(
    gradient_logits,
    data["y_tr"][:512],
)

gradient_loss.backward()

for name, parameter in model.named_parameters():
    grad_norm = None if parameter.grad is None else parameter.grad.norm().item()

    print(f"Gradient {name}: {grad_norm:.6f}")

    assert grad_norm is not None
    assert grad_norm > 0.0


# 6. Thu qua khop 20 mau
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

tiny_model = MLP(
    hidden=(256, 128),
    dropout=0.0,
    init="he",
).to(device)

tiny_optimizer = torch.optim.Adam(
    tiny_model.parameters(),
    lr=1e-2,
)

tiny_X = data["X_tr"][:20]
tiny_y = data["y_tr"][:20]
tiny_losses = []

tiny_model.train()

for step in range(501):
    tiny_optimizer.zero_grad(set_to_none=True)

    tiny_logits = tiny_model(tiny_X)
    tiny_loss = F.cross_entropy(
        tiny_logits,
        tiny_y,
    )

    tiny_losses.append(tiny_loss.item())

    if step == 500:
        break

    tiny_loss.backward()
    tiny_optimizer.step()


tiny_model.eval()

with torch.no_grad():
    tiny_predictions = tiny_model(tiny_X).argmax(dim=1)

    tiny_accuracy = (tiny_predictions == tiny_y).float().mean().item()

print(
    f"Overfit 20 samples: "
    f"loss={tiny_losses[-1]:.6f}, "
    f"accuracy={tiny_accuracy:.1%}"
)

assert tiny_losses[-1] < 0.02
assert tiny_accuracy == 1.0


# 7. Luu duong cong overfit
figure_path = f"{OUT_DIR}/figures/part1_overfit20.png"

plt.figure(figsize=(7, 4))
plt.plot(tiny_losses)
plt.yscale("log")
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.title("Part 1 health check: overfit 20 samples")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(
    figure_path,
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("Saved figure:", figure_path)
print("Part 1: PASS")

**Nhận xét Part 1:** M-base có đúng 47.879 tham số và biến đổi đầu vào `(8, 54)` thành logits `(8, 7)`. Độ lệch chuẩn sau ba lớp Linear lần lượt là 0,6962; 0,6614; 0,6522 nên tín hiệu chưa bị tiêu biến hay bùng nổ.

Loss bước 0 là 2,3776, cao hơn `ln(7) = 1,9459` khoảng 0,4317. Nguyên nhân là khởi tạo He tạo logits có độ phân tán đáng kể ngay từ đầu, khiến mô hình đưa ra một số dự đoán ngẫu nhiên tương đối tự tin thay vì phân phối đều hoàn toàn. Mọi tham số đều nhận gradient khác 0.

Mô hình quá khớp thành công 20 mẫu: loss giảm còn khoảng 0,000001 và accuracy đạt 100%. Điều này cho thấy kiến trúc, nhãn, hàm mất mát, lan truyền ngược và cập nhật tham số đang hoạt động đúng.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# ===== Smoke test: kiem tra pipeline trong 1 epoch =====
from plots import plot_run
from results_table import (
    save_result,
    to_row,
    write_xlsx,
)
from train import DEFAULT_CFG, run_experiment


smoke_cfg = {
    **DEFAULT_CFG,
    "exp_id": "smoke-1ep",
    "group": "smoke",
    "description": "Pipeline smoke test, 1 epoch",
    "lr": 0.05,
    "epochs": 1,
    "seed": 42,
}

smoke_result = run_experiment(
    smoke_cfg,
    data,
)

smoke_dir = f"{OUT_DIR}/smoke"
os.makedirs(smoke_dir, exist_ok=True)

smoke_json = save_result(
    smoke_result,
    results_dir=smoke_dir,
)

smoke_figure = f"{smoke_dir}/smoke-1ep.png"

plot_run(
    smoke_result,
    smoke_figure,
)

smoke_row = to_row(
    smoke_result,
    notes="Chi dung de kiem tra pipeline",
)

smoke_xlsx = f"{smoke_dir}/experiments_smoke.xlsx"

write_xlsx(
    [smoke_row],
    template_path=(
        f"{REPO_ROOT}/templates/"
        "experiment_table_template.xlsx"
    ),
    out_path=smoke_xlsx,
)

assert os.path.isfile(smoke_json)
assert os.path.isfile(smoke_figure)
assert os.path.isfile(smoke_xlsx)

summary = smoke_result["summary"]

print("Smoke summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

assert not summary["diverged"]
assert summary["best_epoch"] in (0, 1)
assert np.isfinite(summary["final_train_loss"])
assert np.isfinite(summary["final_val_loss"])

print("Smoke test: PASS")

In [ ]:
# ===== Tim learning rate cho baseline bang validation =====
import json

from IPython.display import Image, display
from plots import plot_compare, plot_run
from results_table import save_result
from train import DEFAULT_CFG, run_experiment

lr_candidates = [0.01, 0.03, 0.1]
lr_search_results = []
lr_search_summary = []

tuning_dir = f"{OUT_DIR}/tuning_lr"
os.makedirs(tuning_dir, exist_ok=True)

for learning_rate in lr_candidates:
    exp_id = f"tune-sgd-lr{learning_rate:g}"

    cfg = {
        **DEFAULT_CFG,
        "exp_id": exp_id,
        "group": "lr-search",
        "description": (f"SGD momentum learning-rate search, " f"lr={learning_rate}"),
        "lr": learning_rate,
        "epochs": 5,
        "seed": 42,
    }

    result = run_experiment(cfg, data)
    lr_search_results.append(result)

    save_result(
        result,
        results_dir=tuning_dir,
    )

    plot_run(
        result,
        f"{tuning_dir}/{exp_id}.png",
    )

    best_macro_f1 = max(result["history"]["val_macro_f1"])

    best_macro_epoch = result["history"]["val_macro_f1"].index(best_macro_f1) + 1

    lr_search_summary.append(
        {
            "lr": learning_rate,
            "best_val_macro_f1": best_macro_f1,
            "best_macro_epoch": best_macro_epoch,
            "best_val_loss": result["summary"]["best_val_loss"],
            "diverged": result["summary"]["diverged"],
        }
    )


valid_candidates = [item for item in lr_search_summary if not item["diverged"]]

best_candidate = max(
    valid_candidates,
    key=lambda item: item["best_val_macro_f1"],
)

best_lr = best_candidate["lr"]

print("\nLearning-rate search:")
print("lr       best val macro-F1    epoch    best val loss")

for item in lr_search_summary:
    print(
        f"{item['lr']:<8g} "
        f"{item['best_val_macro_f1']:<20.4f} "
        f"{item['best_macro_epoch']:<8d} "
        f"{item['best_val_loss']:.4f}"
    )

print(f"\nSelected baseline learning rate: {best_lr}")

summary_path = f"{tuning_dir}/lr_search_summary.json"

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        lr_search_summary,
        file,
        indent=2,
    )

comparison_path = f"{tuning_dir}/compare_lr_macro_f1.png"

plot_compare(
    lr_search_results,
    metric="val_macro_f1",
    path=comparison_path,
    title=("Baseline LR search: " "validation macro-F1"),
)

display(Image(filename=comparison_path))

print("LR search: PASS")

**Dự đoán baseline:** Với SGD momentum, `lr=0.1` được chọn bằng validation và 20 epoch, train/validation loss dự kiến giảm ổn định; accuracy phải vượt rõ mốc lớp đa số 0,4876. Kết quả giữa các seed sẽ dao động nhẹ, vì vậy cần dùng trung bình ± độ lệch chuẩn trước khi kết luận cấu hình khác tốt hơn baseline.

In [ ]:
# ===== Baseline 20 epoch, 3 seed =====
from IPython.display import Image, display

from plots import plot_compare, plot_run
from results_table import save_result, to_row, write_xlsx
from train import DEFAULT_CFG, run_experiment

baseline_lr = 0.1
baseline_seeds = [1, 42, 2026]
baseline_results = []

for seed in baseline_seeds:
    exp_id = f"base-s{seed}"

    cfg = {
        **DEFAULT_CFG,
        "exp_id": exp_id,
        "group": "baseline",
        "description": (
            "M-base, CE, SGD momentum 0.9, " f"lr={baseline_lr}, seed={seed}"
        ),
        "lr": baseline_lr,
        "epochs": 20,
        "seed": seed,
    }

    result = run_experiment(cfg, data)
    baseline_results.append(result)

    save_result(
        result,
        results_dir=f"{OUT_DIR}/results",
    )

    plot_run(
        result,
        path=f"{OUT_DIR}/figures/{exp_id}.png",
    )


baseline_by_seed = {result["cfg"]["seed"]: result for result in baseline_results}

# Seed 42 la baseline tham chieu cho cac thi nghiem sau
baseline_result = baseline_by_seed[42]

baseline_acc = np.array([result["summary"]["val_acc"] for result in baseline_results])

baseline_f1 = np.array(
    [result["summary"]["val_macro_f1"] for result in baseline_results]
)

baseline_loss = np.array(
    [result["summary"]["best_val_loss"] for result in baseline_results]
)

noise_summary = {
    "seeds": baseline_seeds,
    "val_acc_mean": float(baseline_acc.mean()),
    "val_acc_std": float(baseline_acc.std(ddof=1)),
    "val_macro_f1_mean": float(baseline_f1.mean()),
    "val_macro_f1_std": float(baseline_f1.std(ddof=1)),
    "best_val_loss_mean": float(baseline_loss.mean()),
    "best_val_loss_std": float(baseline_loss.std(ddof=1)),
}

print("\nBaseline seed results:")
print("seed   best epoch   val acc   val macro-F1   val loss")

for result in baseline_results:
    summary = result["summary"]

    print(
        f"{result['cfg']['seed']:<6d} "
        f"{summary['best_epoch']:<12d} "
        f"{summary['val_acc']:<9.4f} "
        f"{summary['val_macro_f1']:<14.4f} "
        f"{summary['best_val_loss']:.4f}"
    )

print(
    "\nValidation accuracy: "
    f"{noise_summary['val_acc_mean']:.4f} "
    f"± {noise_summary['val_acc_std']:.4f}"
)

print(
    "Validation macro-F1: "
    f"{noise_summary['val_macro_f1_mean']:.4f} "
    f"± {noise_summary['val_macro_f1_std']:.4f}"
)

with open(
    f"{OUT_DIR}/results/baseline_noise.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        noise_summary,
        file,
        indent=2,
    )


compare_f1_path = f"{OUT_DIR}/figures/" "compare_baseline_seeds_f1.png"

plot_compare(
    baseline_results,
    metric="val_macro_f1",
    path=compare_f1_path,
    title="Baseline seed noise: validation macro-F1",
)

compare_loss_path = f"{OUT_DIR}/figures/" "compare_baseline_seeds_loss.png"

plot_compare(
    baseline_results,
    metric="val_loss",
    path=compare_loss_path,
    title="Baseline seed noise: validation loss",
)


baseline_rows = [
    to_row(
        result,
        notes=("Baseline seed repeat; " "lr selected using validation"),
    )
    for result in baseline_results
]

write_xlsx(
    baseline_rows,
    template_path=(f"{REPO_ROOT}/templates/" "experiment_table_template.xlsx"),
    out_path=f"{OUT_DIR}/experiments.xlsx",
)

display(Image(filename=compare_f1_path))
display(Image(filename=(f"{OUT_DIR}/figures/base-s42.png")))

print("Baseline 3 seeds: PASS")

**Nhận xét baseline:** Learning rate `0,1` được chọn bằng validation sau khi thử `0,01`, `0,03` và `0,1`. Với 20 epoch, cả ba seed đều vượt xa mốc đoán lớp đa số 0,4876.

Kết quả tốt nhất theo validation loss:

- Seed 1: best epoch 18, val accuracy 0,9054, val macro-F1 0,8390, val loss 0,2379.
- Seed 42: best epoch 20, val accuracy 0,9059, val macro-F1 0,8564, val loss 0,2380.
- Seed 2026: best epoch 20, val accuracy 0,9086, val macro-F1 0,8496, val loss 0,2302.

Trung bình validation accuracy là `0,9066 ± 0,0017`; validation macro-F1 là `0,8484 ± 0,0087`. Vì vậy, chênh lệch macro-F1 nhỏ hơn khoảng `2σ = 0,0174` chưa đủ mạnh để khẳng định một cấu hình tốt hơn baseline.

Train loss và validation loss nhìn chung cùng giảm; khoảng cách ở seed 42 tại epoch 20 nhỏ (`0,2380 - 0,2172 ≈ 0,0208`), chưa có dấu hiệu quá khớp rõ. Macro-F1 vẫn tăng về cuối quá trình huấn luyện. Gradient norm ổn định khoảng 0,51–0,58, không có hiện tượng bùng nổ gradient.

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
### Thí nghiệm 1 — Bộ tối ưu

**Dự đoán trước:** Adam có thể hội tụ nhanh hơn SGD Momentum trong các epoch đầu nhờ learning rate thích nghi theo từng tham số. Tuy nhiên, kết quả cuối có thể chỉ chênh nhẹ và nằm trong nhiễu giữa các seed. Tôi giữ nguyên kiến trúc, loss, batch size, seed, số epoch và cách khởi tạo; mỗi optimizer được đánh giá với ít nhất hai learning rate.

In [ ]:
# ===== Part 3.1: optimizer =====
import json

from IPython.display import Image, display
from plots import plot_compare, plot_run
from results_table import load_results, save_result, to_row, write_xlsx
from train import DEFAULT_CFG, run_experiment

# Baseline seed 42, SGD Momentum lr=0.1 la ung vien thu nhat cua SGD Momentum.
with open(
    f"{OUT_DIR}/results/base-s42.json",
    "r",
    encoding="utf-8",
) as file:
    baseline_reference = json.load(file)

optimizer_specs = [
    {
        "exp_id": "opt-sgdm-lr0.03",
        "optimizer": "sgd_momentum",
        "lr": 0.03,
    },
    {
        "exp_id": "opt-adam-lr3e-4",
        "optimizer": "adam",
        "lr": 3e-4,
    },
    {
        "exp_id": "opt-adam-lr1e-3",
        "optimizer": "adam",
        "lr": 1e-3,
    },
]

optimizer_results = []

for spec in optimizer_specs:
    cfg = {
        **DEFAULT_CFG,
        "exp_id": spec["exp_id"],
        "group": "optimizer",
        "description": (
            f"M-base, CE, {spec['optimizer']}, " f"lr={spec['lr']}, seed=42"
        ),
        "optimizer": spec["optimizer"],
        "lr": spec["lr"],
        "epochs": 20,
        "seed": 42,
    }

    result = run_experiment(cfg, data)
    optimizer_results.append(result)

    save_result(
        result,
        results_dir=f"{OUT_DIR}/results",
    )

    plot_run(
        result,
        path=f"{OUT_DIR}/figures/{cfg['exp_id']}.png",
    )

comparison_results = [baseline_reference, *optimizer_results]
baseline_f1 = baseline_reference["summary"]["val_macro_f1"]

print("\nOptimizer results:")
print(
    "exp_id               optimizer       lr       epoch   val_acc   macro-F1   delta"
)

for result in comparison_results:
    cfg = result["cfg"]
    summary = result["summary"]

    print(
        f"{cfg['exp_id']:<20} "
        f"{cfg['optimizer']:<15} "
        f"{cfg['lr']:<8g} "
        f"{summary['best_epoch']:<7d} "
        f"{summary['val_acc']:<9.4f} "
        f"{summary['val_macro_f1']:<10.4f} "
        f"{summary['val_macro_f1'] - baseline_f1:+.4f}"
    )

sgdm_results = [
    result
    for result in comparison_results
    if result["cfg"]["optimizer"] == "sgd_momentum"
]

adam_results = [
    result for result in comparison_results if result["cfg"]["optimizer"] == "adam"
]

best_sgdm = max(
    sgdm_results,
    key=lambda result: result["summary"]["val_macro_f1"],
)

best_adam = max(
    adam_results,
    key=lambda result: result["summary"]["val_macro_f1"],
)

with open(
    f"{OUT_DIR}/results/baseline_noise.json",
    "r",
    encoding="utf-8",
) as file:
    noise_summary = json.load(file)

noise_threshold = 2 * noise_summary["val_macro_f1_std"]
optimizer_delta = (
    best_adam["summary"]["val_macro_f1"] - best_sgdm["summary"]["val_macro_f1"]
)

print("\nSelected by validation macro-F1:")
print(
    f"Best SGD Momentum: {best_sgdm['cfg']['exp_id']} | "
    f"F1={best_sgdm['summary']['val_macro_f1']:.4f}"
)
print(
    f"Best Adam:         {best_adam['cfg']['exp_id']} | "
    f"F1={best_adam['summary']['val_macro_f1']:.4f}"
)
print(f"Adam - SGD Momentum: {optimizer_delta:+.4f}")
print(f"Baseline 2-sigma threshold: {noise_threshold:.4f}")

compare_path = f"{OUT_DIR}/figures/compare_optimizer_f1.png"

plot_compare(
    comparison_results,
    metric="val_macro_f1",
    path=compare_path,
    title="Optimizer comparison: validation macro-F1",
)

all_results = load_results(f"{OUT_DIR}/results")

rows = [
    to_row(
        result,
        notes=(
            "Optimizer comparison; seed=42; " "other settings follow baseline"
            if result["cfg"]["group"] == "optimizer"
            else "Baseline seed repeat; lr selected using validation"
        ),
    )
    for result in all_results
]

write_xlsx(
    rows,
    template_path=(f"{REPO_ROOT}/templates/" "experiment_table_template.xlsx"),
    out_path=f"{OUT_DIR}/experiments.xlsx",
)

display(Image(filename=compare_path))
print("Optimizer experiment: PASS")

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary